# 15.2 推理能力评估 (Reasoning Evaluation)

评估模型的逻辑推理和问题解决能力，是大模型能力评估的核心维度。

本节涵盖：
- 数学推理评估（GSM8K/MATH）
- 代码生成评估（pass@k）
- Chain-of-Thought评估
- LLM-as-Judge

## 1. 数学推理评估

**目的**：评估模型的多步数学推理能力

**核心基准**：
- **GSM8K**：8500+小学数学题，需要2-8步推理，评估基础数学能力
- **MATH**：5000+竞赛级数学题，涵盖代数/几何/数论/概率等，评估高阶数学能力

**评估指标**：
- **Exact Match**：最终答案完全匹配（GSM8K主要指标）
- **过程评分**：中间步骤的正确性（更细粒度但评估成本高）

**关键技巧**：
- 答案提取：从模型输出中提取最终数字答案（正则表达式）
- 等价性判断："1/2"="0.5"="50%"，需要规范化
- 多数投票（Majority Voting）：采样多次取多数答案，显著提升准确率

In [ ]:
import re
import torch
import torch.nn.functional as F
from collections import Counter

torch.manual_seed(42)

def extract_answer(text):
    patterns = [
        r'The answer is ([-\d./]+)',
        r'\\boxed\{([-\d./]+)\}',
        r'=\s*\\?([-\d./]+)\s*$',
        r'answer:?\s*([-\d./]+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1)
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else ''

def normalize_answer(ans):
    ans = ans.strip().replace(',', '')
    try:
        if '/' in ans:
            parts = ans.split('/')
            result = float(parts[0]) / float(parts[1])
            return f'{result:.4f}'
        if '%' in ans:
            return f'{float(ans.replace("%", "")) / 100:.4f}'
        return f'{float(ans):.4f}'
    except (ValueError, ZeroDivisionError):
        return ans

def exact_match(pred, target):
    return normalize_answer(pred) == normalize_answer(target)

def majority_vote(answers):
    normalized = [normalize_answer(a) for a in answers]
    counter = Counter(normalized)
    return counter.most_common(1)[0][0]

print('=== Math Reasoning Evaluation ===')

sample_outputs = [
    ('To solve 3x + 5 = 20, we subtract 5 to get 3x = 15, then divide by 3. The answer is 5', '5'),
    ('After calculating, \\boxed{0.5}', '0.5'),
    ('The result equals 1/2', '0.5'),
    ('50% of the total', '0.5'),
    ('The answer is 3.14159', '3.14'),
]

print('Answer extraction and normalization:')
for output, target in sample_outputs:
    extracted = extract_answer(output)
    normalized = normalize_answer(extracted)
    target_norm = normalize_answer(target)
    match = normalized == target_norm
    print(f'  Output: "{output[:50]}..."')
    print(f'  Extracted: {extracted} → Normalized: {normalized} vs Target: {target_norm} → {"✓" if match else "✗"}')

print(f'\nMajority voting demo:')
vote_answers = ['5', '5', '5.0', '4', '5', '5.00']
winner = majority_vote(vote_answers)
print(f'  Answers: {vote_answers}')
print(f'  Majority vote result: {winner}')

print(f'\nKey: Answer extraction is critical for automated evaluation.')
print(f'Normalization handles format variations (fractions, percentages, decimals).')
print(f'Majority voting with n>1 samples significantly improves accuracy.')

## 2. 代码生成评估（pass@k）

**目的**：评估模型生成正确代码的能力

**核心基准**：
- **HumanEval**：164个Python编程题，每个题有单元测试
- **MBPP**：974个基础编程题，覆盖更广的Python功能
- **SWE-bench**：真实GitHub issue修复，评估工程能力

**pass@k指标**：
- 生成n个代码样本，如果至少1个通过测试，则pass@n=1
- pass@k = 1 - C(n-c, k) / C(n, k)，其中c是正确样本数
- **pass@1**：单次生成正确率（最实用，生产环境只生成1次）
- **pass@10**：10次中至少1次正确的概率（评估潜力）
- **pass@100**：100次中至少1次正确的概率（理论上限）

**温度与pass@k的关系**：
- T=0（贪心）：pass@1最高，但pass@10/100低
- T=0.8：pass@1稍低，但pass@10/100显著更高
- 工程实践：生产用T=0.1-0.3，评估用T=0.8

In [ ]:
import math
from scipy.special import comb as comb_choose

def pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    return 1.0 - comb_choose(n - c, k) / comb_choose(n, k)

def pass_at_k_approx(n, c, k):
    if n - c < k:
        return 1.0
    return 1.0 - ((n - c) / n) ** k

print('=== Code Generation Evaluation: pass@k ===')

print('\nExample: n=10 samples generated, varying number correct (c)')
print(f'{"c":>3} {"pass@1":>8} {"pass@5":>8} {"pass@10":>8}')
for c in range(0, 11):
    p1 = pass_at_k(10, c, 1)
    p5 = pass_at_k(10, c, 5)
    p10 = pass_at_k(10, c, 10)
    print(f'{c:>3} {p1:>8.3f} {p5:>8.3f} {p10:>8.3f}')

print('\n--- Temperature vs pass@k (simulated) ---')
n_samples = 200
results = {
    'T=0.0 (greedy)': {'c': 80, 'n': n_samples},
    'T=0.4': {'c': 90, 'n': n_samples},
    'T=0.8': {'c': 100, 'n': n_samples},
    'T=1.0': {'c': 95, 'n': n_samples},
}
print(f'{"Setting":>18} {"pass@1":>8} {"pass@10":>8} {"pass@100":>8}')
for name, r in results.items():
    p1 = pass_at_k(r['n'], r['c'], 1)
    p10 = pass_at_k(r['n'], r['c'], 10)
    p100 = pass_at_k(r['n'], r['c'], 100)
    print(f'{name:>18} {p1:>8.3f} {p10:>8.3f} {p100:>8.3f}')

print('\n--- Code Execution Safety ---')
print('Production code evaluation requires sandboxing:')
print('  1. Docker containers with resource limits (CPU, memory, time)')
print('  2. Network isolation (no internet access)')
print('  3. Filesystem isolation (read-only, no /tmp writes)')
print('  4. Timeout enforcement (5-30 seconds per test)')
print('  5. Import restrictions (no os, subprocess, socket, etc.)')

print(f'\nKey: pass@1 is the most practical metric for production systems.')
print(f'Higher temperature increases diversity (better pass@k for k>1) but may decrease pass@1.')
print(f'Sandboxed execution is mandatory for safe code evaluation.')

## 3. Chain-of-Thought评估

**目的**：评估模型推理过程的正确性，而不仅仅是最终答案

**评估维度**：
- **答案正确性**：最终答案是否正确
- **过程正确性**：中间推理步骤是否正确
- **步骤完整性**：是否遗漏关键步骤
- **逻辑一致性**：步骤之间是否自洽

**评估方法**：
- **自动评估**：用更强的模型（如GPT-4）评判推理步骤
- **过程奖励模型（PRM）**：训练模型对每步推理打分
- **人工评估**：专家逐步检查，成本高但最可靠

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class StepRewardModel(nn.Module):
    def __init__(self, d_model=256, n_heads=4, n_layers=2, vocab_size=1000):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(512, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers)
        self.step_scorer = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, step_ids, step_positions):
        h = self.embed(step_ids) + self.pos_embed(step_positions)
        h = self.transformer(h)
        step_scores = self.step_scorer(h).squeeze(-1)
        return step_scores

prm = StepRewardModel()

n_steps = 5
seq_len = 20
step_ids = torch.randint(0, 1000, (1, n_steps * seq_len))
step_positions = torch.arange(n_steps * seq_len).unsqueeze(0)

step_scores = prm(step_ids, step_positions)

step_score_list = [step_scores[0, i*seq_len:(i+1)*seq_len].mean().item() for i in range(n_steps)]

print('=== Chain-of-Thought Step-by-Step Evaluation ===')
print(f'\nProcess Reward Model (PRM) scores per step:')
steps = [
    'Step 1: Read the problem and identify variables',
    'Step 2: Set up the equation 3x + 5 = 20',
    'Step 3: Subtract 5 from both sides: 3x = 15',
    'Step 4: Divide by 3: x = 5',
    'Step 5: Verify: 3(5) + 5 = 20 ✓',
]
for i, (step, score) in enumerate(zip(steps, step_score_list)):
    quality = '✓ Correct' if score > 0 else '✗ Error'
    print(f'  Step {i+1}: score={score:.3f} {quality}')
    print(f'    "{step}"')

total_score = sum(step_score_list) / len(step_score_list)
print(f'\nOverall process score: {total_score:.3f}')

print(f'\n--- CoT Evaluation Methods Comparison ---')
methods = [
    ('Outcome RM', 'Only scores final answer', 'Cheap, misses process errors'),
    ('Process RM', 'Scores each step', 'Catches process errors, needs step-level labels'),
    ('LLM-as-Judge', 'GPT-4 evaluates steps', 'Flexible, expensive, may be biased'),
    ('Human Eval', 'Expert reviews steps', 'Gold standard, very expensive'),
]
print(f'{"Method":>15} {"Description":>30} {"Trade-off":>40}')
for name, desc, tradeoff in methods:
    print(f'{name:>15} {desc:>30} {tradeoff:>40}')

print(f'\nKey: Process Reward Models (PRM) provide step-level feedback.')
print(f'PRM is used in OpenAI o1 and similar reasoning models for process supervision.')
print(f'Outcome RM is simpler but cannot distinguish correct process from lucky guesses.')

## 4. LLM-as-Judge

**目的**：用强模型评估弱模型的输出质量

**基本原理**：使用GPT-4/Claude等强模型作为评判者，对被评估模型的输出打分。适用于没有标准答案的开放性任务（创意写作、对话质量等）。

**评估模式**：
- **单点评分**：给一个输出打1-5分
- **成对比较**：两个输出哪个更好（更稳定）
- **参考对比**：与参考答案比较

**关键问题**：
- **位置偏见**：成对比较中，Judge倾向选择第一个/第二个
- **冗长偏见**：Judge倾向给更长的输出更高分
- **自我偏好**：Judge倾向给自己的风格更高分
- **一致性**：同一输入多次评估结果应一致

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

torch.manual_seed(42)

class LLMJudgeSimulator:
    def __init__(self, position_bias=0.05, verbosity_bias=0.1, self_preference=0.05):
        self.position_bias = position_bias
        self.verbosity_bias = verbosity_bias
        self.self_preference = self_preference

    def score_single(self, quality, length_tokens):
        base_score = quality * 4 + 1
        length_bonus = min(length_tokens / 500, 1.0) * self.verbosity_bias * 5
        noise = random.gauss(0, 0.3)
        return max(1, min(5, base_score + length_bonus + noise))

    def compare_pair(self, quality_a, quality_b, length_a, length_b, swap=False):
        score_a = quality_a + min(length_a / 500, 1.0) * self.verbosity_bias
        score_b = quality_b + min(length_b / 500, 1.0) * self.verbosity_bias
        if swap:
            score_a, score_b = score_b, score_a
            score_a += self.position_bias
        else:
            score_a += self.position_bias
        noise = random.gauss(0, 0.1)
        return 'A' if score_a + noise > score_b else 'B'

judge = LLMJudgeSimulator(position_bias=0.05, verbosity_bias=0.1)

print('=== LLM-as-Judge Evaluation ===')

print('\n--- Single-point Scoring ---')
samples = [
    ('Short correct answer', 0.9, 50),
    ('Long detailed answer', 0.9, 400),
    ('Short wrong answer', 0.3, 50),
    ('Long wrong answer', 0.3, 400),
]
for desc, quality, length in samples:
    score = judge.score_single(quality, length)
    print(f'  {desc:25s}: quality={quality:.1f}, length={length:3d} → score={score:.2f}')

print('\n--- Pairwise Comparison (with position bias detection) ---')
n_trials = 100
a_wins_normal = 0
a_wins_swapped = 0
for _ in range(n_trials):
    result = judge.compare_pair(0.8, 0.8, 200, 200, swap=False)
    if result == 'A':
        a_wins_normal += 1
    result_swap = judge.compare_pair(0.8, 0.8, 200, 200, swap=True)
    if result_swap == 'A':
        a_wins_swapped += 1

print(f'Equal quality, A first: A wins {a_wins_normal}% (position bias)')
print(f'Equal quality, B first: A wins {a_wins_swapped}%')
print(f'Position bias detected: {abs(a_wins_normal - 50):.0f}% deviation from 50%')

print('\n--- Consistency Check ---')
scores = [judge.score_single(0.8, 200) for _ in range(20)]
mean_score = sum(scores) / len(scores)
std_score = (sum((s - mean_score)**2 for s in scores) / len(scores)) ** 0.5
print(f'20 evaluations of same input: mean={mean_score:.2f}, std={std_score:.2f}')
print(f'Consistency: {"Good" if std_score < 0.5 else "Poor"} (std < 0.5)')

print(f'\nKey: LLM-as-Judge is flexible but has biases.')
print(f'Mitigation: swap positions in pairwise comparison, average multiple evaluations.')
print(f'Always report inter-rater agreement (Cohen\'s kappa) for reliability.')